In [14]:
import json
from collections import Counter, defaultdict
from pathlib import Path
import sys

# --- Main Analysis Function ---
def analyze_results_summary(json_path):
    """
    Reads the JSON output file, aggregates warnings into categories,
    calculates reference counts, and prints a concise summary report,
    sorted by counts where applicable.
    """
    if not json_path.is_file():
        print(f"Error: JSON file not found at '{json_path}'", file=sys.stderr)
        return

    print(f"Analyzing results summary in: {json_path}")

    aggregated_warning_counts = Counter()
    references_per_category = defaultdict(int)
    total_files_processed = 0
    files_with_warnings = 0
    total_references_found = 0

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            all_files_data = json.load(f)
        total_files_processed = len(all_files_data)
        print(f"Successfully loaded data for {total_files_processed} files.")
    except Exception as e:
        print(f"Error reading or parsing file '{json_path}': {e}", file=sys.stderr)
        return

    for file_data in all_files_data:
        current_refs = file_data.get("number_of_references", 0)
        if not isinstance(current_refs, int):
            print(f"Warning: 'number_of_references' in '{file_data.get('filename', 'Unknown')}' is not an integer ({type(current_refs)}). Treating as 0.", file=sys.stderr)
            current_refs = 0
        total_references_found += current_refs

        category = file_data.get("category", "unknown")
        if not isinstance(category, str):
            print(f"Warning: 'category' in '{file_data.get('filename', 'Unknown')}' is not a string ({type(category)}). Assigning to 'unknown'.", file=sys.stderr)
            category = "unknown"
        references_per_category[category] += current_refs

        warnings_list = file_data.get("warnings", [])
        if isinstance(warnings_list, list) and warnings_list:
            files_with_warnings += 1
            for warning in warnings_list:
                if "Sequence error: " in warning and "reason: gap too large" in warning:
                    aggregated_warning_counts["Sequence Errors (Gap Too Large)"] += 1
                elif "Sequence error: " in warning and "reason: out of order" in warning:
                    aggregated_warning_counts["Sequence Errors (Out of Order)"] += 1
                elif "skipped on page" in warning and "(X-margin check)" in warning:
                    aggregated_warning_counts["Footnotes Skipped (X-Margin)"] += 1
                elif "Minimum threshold" in warning and "reached" in warning:
                    aggregated_warning_counts["Processing Failures (Min Threshold Reached)"] += 1
                elif warning == "Critical PDF processing error":
                    aggregated_warning_counts["Processing Failures (Critical Error)"] += 1
                elif "Block extraction failed" in warning:
                     aggregated_warning_counts["Processing Failures (Block Extraction)"] += 1
                elif "Number parsing failed" in warning:
                     aggregated_warning_counts["Processing Failures (Number Parsing)"] += 1
                elif "Line processing error" in warning:
                     aggregated_warning_counts["Processing Failures (Line Processing)"] += 1
                elif "Category determination failed" in warning:
                     aggregated_warning_counts["Processing Failures (Category Determination)"] += 1
                elif warning.startswith("Acceptable gap detected:"):
                     pass # Ignoring acceptable gaps
                else:
                    aggregated_warning_counts["Processing Failures (Other/Unknown)"] += 1
        elif not isinstance(warnings_list, list):
             filename = file_data.get("filename", "Unknown Filename")
             print(f"Warning: 'warnings' field in '{filename}' is not a list: {type(warnings_list)}. Skipping warnings for this file.", file=sys.stderr)

    # --- Print Summary Report ---
    print("\n--- Results Analysis Summary ---")
    print(f"Total files analyzed: {total_files_processed}")
    print(f"Files containing logged warnings: {files_with_warnings}")
    print("-" * 40)
    print(f"Total references found across all files: {total_references_found}")
    print("-" * 40)

    print("Aggregated Warning Counts by Type (Highest Count First):")
    if aggregated_warning_counts:
        # Sort by count (item[1]) in descending order (reverse=True)
        for category, count in sorted(aggregated_warning_counts.items(), key=lambda item: item[1], reverse=True):
            print(f"{category:<45} : {count:>6}")
    else:
        print("No aggregated warnings recorded.")
    print("-" * 40)

    print("Total References per Category (Highest Count First):")
    if references_per_category:
         # Sort by count (item[1]) in descending order (reverse=True)
        for category, count in sorted(references_per_category.items(), key=lambda item: item[1], reverse=True):
             print(f"{category:<80} : {count:>6}")
    else:
        print("No category reference data available.")
    print("-" * 40)

In [16]:
# --- Configuration ---
JSON_FILE_PATH = Path("swiss_law_footers_pdf_dynamic_margin.json") # Make sure this points to the correct input JSON

if __name__ == "__main__":
    analyze_results_summary(JSON_FILE_PATH)

Analyzing results summary in: swiss_law_footers_pdf_dynamic_margin.json
Successfully loaded data for 21641 files.

--- Results Analysis Summary ---
Total files analyzed: 21641
Files containing logged warnings: 7576
----------------------------------------
Total references found across all files: 46479
----------------------------------------
Aggregated Warning Counts by Type (Highest Count First):
Footnotes Skipped (X-Margin)                  :  24079
Sequence Errors (Gap Too Large)               :  14425
Sequence Errors (Out of Order)                :  13606
Processing Failures (Min Threshold Reached)   :    903
----------------------------------------
Total References per Category (Highest Count First):
Ordinanza del Consiglio federale                                                 :  18474
Ordinanza di un dipartimento                                                     :   6328
Legge federale                                                                   :   6222
Ordinanza di un

In [17]:
# --- Configuration ---
JSON_FILE_PATH = Path("0.85_swiss_law_footers_pdf_min_x0_margin.json") # Make sure this points to the correct input JSON

if __name__ == "__main__":
    analyze_results_summary(JSON_FILE_PATH)

Analyzing results summary in: 0.85_swiss_law_footers_pdf_min_x0_margin.json
Successfully loaded data for 21641 files.

--- Results Analysis Summary ---
Total files analyzed: 21641
Files containing logged warnings: 9414
----------------------------------------
Total references found across all files: 66779
----------------------------------------
Aggregated Warning Counts by Type (Highest Count First):
Sequence Errors (Gap Too Large)               :  14512
Footnotes Skipped (X-Margin)                  :  12010
Sequence Errors (Out of Order)                :   8226
Processing Failures (Min Threshold Reached)   :   3163
----------------------------------------
Total References per Category (Highest Count First):
Ordinanza del Consiglio federale                                                 :  24018
Ordinanza di un dipartimento                                                     :  10084
Legge federale                                                                   :   8364
Ordinanza d

In [20]:
# --- Configuration ---
JSON_FILE_PATH = Path("0.75_swiss_law_footers_pdf_min_x0_margin.json") # Make sure this points to the correct input JSON

if __name__ == "__main__":
    analyze_results_summary(JSON_FILE_PATH)

Analyzing results summary in: 0.75_swiss_law_footers_pdf_min_x0_margin.json
Successfully loaded data for 21641 files.

--- Results Analysis Summary ---
Total files analyzed: 21641
Files containing logged warnings: 11831
----------------------------------------
Total references found across all files: 82397
----------------------------------------
Aggregated Warning Counts by Type (Highest Count First):
Sequence Errors (Out of Order)                :  27705
Sequence Errors (Gap Too Large)               :  16972
Footnotes Skipped (X-Margin)                  :  16260
Processing Failures (Min Threshold Reached)   :   3504
----------------------------------------
Total References per Category (Highest Count First):
Ordinanza del Consiglio federale                                                 :  31854
Legge federale                                                                   :  12024
Ordinanza di un dipartimento                                                     :  11461
Ordinanza 

In [21]:
# --- Configuration ---
JSON_FILE_PATH = Path("0.70_swiss_law_footers_pdf_min_x0_margin.json") # Make sure this points to the correct input JSON

if __name__ == "__main__":
    analyze_results_summary(JSON_FILE_PATH)

Analyzing results summary in: 0.70_swiss_law_footers_pdf_min_x0_margin.json
Successfully loaded data for 21641 files.

--- Results Analysis Summary ---
Total files analyzed: 21641
Files containing logged warnings: 12961
----------------------------------------
Total references found across all files: 85576
----------------------------------------
Aggregated Warning Counts by Type (Highest Count First):
Sequence Errors (Out of Order)                :  39950
Sequence Errors (Gap Too Large)               :  19666
Footnotes Skipped (X-Margin)                  :  19012
Processing Failures (Min Threshold Reached)   :   3798
----------------------------------------
Total References per Category (Highest Count First):
Ordinanza del Consiglio federale                                                 :  33482
Legge federale                                                                   :  12410
Ordinanza di un dipartimento                                                     :  12005
Ordinanza 

In [22]:
# --- Configuration ---
JSON_FILE_PATH = Path("0.60_swiss_law_footers_pdf_min_x0_margin.json") # Make sure this points to the correct input JSON

if __name__ == "__main__":
    analyze_results_summary(JSON_FILE_PATH)

Analyzing results summary in: 0.60_swiss_law_footers_pdf_min_x0_margin.json
Successfully loaded data for 21641 files.

--- Results Analysis Summary ---
Total files analyzed: 21641
Files containing logged warnings: 15060
----------------------------------------
Total references found across all files: 89626
----------------------------------------
Aggregated Warning Counts by Type (Highest Count First):
Sequence Errors (Out of Order)                :  65622
Sequence Errors (Gap Too Large)               :  29437
Footnotes Skipped (X-Margin)                  :  24008
Processing Failures (Min Threshold Reached)   :   4262
----------------------------------------
Total References per Category (Highest Count First):
Ordinanza del Consiglio federale                                                 :  35438
Ordinanza di un dipartimento                                                     :  12747
Legge federale                                                                   :  12595
Ordinanza 

# Let's extract the precise law codes from the references

In [24]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
# Input JSON file from the previous step
INPUT_JSON_PATH = Path("0.60_swiss_law_footers_pdf_min_x0_margin.json")
# Output JSON file with the added reference codes
OUTPUT_JSON_PATH = Path("swiss_law_footers_with_codes.json")

# --- Regex Definitions ---
# Compile regex patterns for efficiency

# Pattern for FF YYYY NNNN or RU YYYY NNNN formats
# \b ensures match is a whole word/code
# (?:FF|RU) matches FF or RU non-capturingly
# \s+ matches one or more spaces
# \d{4} matches a 4-digit year
# \d+ matches the final number part
FF_RU_PATTERN = re.compile(r"\b(?:FF|RU)\s+\d{4}\s+\d+\b")

# Pattern for RS N.N.N... or RS NNN formats
# \bRS matches RS at word start
# \s+ matches one or more spaces
# \d+ matches the initial digit(s)
# (?:\.\d+)* optionally matches groups of '.' followed by digits zero or more times
# \b ensures match is a whole word/code
RS_PATTERN = re.compile(r"\bRS\s+\d+(?:\.\d+)*\b")

# --- Main Processing Function ---
def add_reference_codes(input_path, output_path):
    """
    Reads the input JSON, extracts law reference codes from footnote text,
    adds them as a new list to each file object, and saves to a new JSON file.
    """
    # Check if input file exists
    if not input_path.is_file():
        print(f"Error: Input JSON file not found at '{input_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_path}")
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            all_files_data = json.load(f)
        print(f"Loaded data for {len(all_files_data)} files.")
    except Exception as e:
        print(f"Error reading or parsing input JSON file '{input_path}': {e}", file=sys.stderr)
        return

    print("Processing files to extract reference codes...")
    processed_count = 0
    # Iterate through each file's data structure in the list
    for file_data in all_files_data:
        # Initialize a set to store unique codes found for this file
        codes_found_in_file = set()
        filename = file_data.get("filename", "Unknown Filename") # For logging

        # Get the list of references (handle potential missing key)
        references_list = file_data.get("references", [])

        # Check if references_list is actually a list
        if not isinstance(references_list, list):
            print(f"Warning: 'references' field in '{filename}' is not a list. Skipping code extraction for this file.", file=sys.stderr)
            # Ensure the output key exists, even if empty
            file_data["reference_codes"] = []
            continue # Move to the next file

        # Iterate through each reference dictionary in the list
        for ref_dict in references_list:
             # Ensure ref_dict is a dictionary
             if not isinstance(ref_dict, dict):
                 print(f"Warning: Item in 'references' for '{filename}' is not a dictionary: {ref_dict}. Skipping.", file=sys.stderr)
                 continue

             # Extract the footnote text (value of the first key, assuming single key-value pair)
             try:
                 # Get dictionary values, convert to list, take the first element
                 footnote_text = list(ref_dict.values())[0]
                 # Ensure the extracted text is a string
                 if not isinstance(footnote_text, str):
                     print(f"Warning: Footnote text in '{filename}' is not a string: {type(footnote_text)}. Skipping.", file=sys.stderr)
                     continue # Skip this reference if text is not a string
             except IndexError:
                 # Handle case where dictionary might be empty
                 print(f"Warning: Empty reference dictionary found in '{filename}'. Skipping.", file=sys.stderr)
                 continue # Skip this empty reference dictionary
             except Exception as e:
                 # Catch other potential errors during text extraction
                 print(f"Warning: Error extracting footnote text from {ref_dict} in '{filename}': {e}. Skipping.", file=sys.stderr)
                 continue # Skip this problematic reference

             # --- Find and Add Codes ---
             # Find all matches for the FF/RU pattern in the text
             ff_ru_matches = FF_RU_PATTERN.findall(footnote_text)
             # Add found matches to the set (set handles uniqueness automatically)
             codes_found_in_file.update(ff_ru_matches)

             # Find all matches for the RS pattern in the text
             rs_matches = RS_PATTERN.findall(footnote_text)
             # Add found matches to the set
             codes_found_in_file.update(rs_matches)

        # --- Add Results to File Data ---
        # Convert the set of unique codes to a sorted list
        sorted_codes = sorted(list(codes_found_in_file))
        # Add the list to the file's data structure
        file_data["reference_codes"] = sorted_codes
        processed_count += 1
        if processed_count % 1000 == 0: # Log progress every 1000 files
             print(f"Processed {processed_count}/{len(all_files_data)} files...")

    print(f"Finished processing {processed_count} files.")

    # --- Save Updated Data ---
    print(f"Saving updated data with reference codes to: {output_path}")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            # Dump the modified list of dictionaries back to JSON
            json.dump(all_files_data, f, ensure_ascii=False, indent=4)
        print("Successfully saved updated JSON file.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_path}': {e}", file=sys.stderr)


# --- Script Execution ---
if __name__ == "__main__":
    add_reference_codes(INPUT_JSON_PATH, OUTPUT_JSON_PATH)

Loading data from: 0.60_swiss_law_footers_pdf_min_x0_margin.json
Loaded data for 21641 files.
Processing files to extract reference codes...
Processed 1000/21641 files...
Processed 2000/21641 files...
Processed 3000/21641 files...
Processed 4000/21641 files...
Processed 5000/21641 files...
Processed 6000/21641 files...
Processed 7000/21641 files...
Processed 8000/21641 files...
Processed 9000/21641 files...
Processed 10000/21641 files...
Processed 11000/21641 files...
Processed 12000/21641 files...
Processed 13000/21641 files...
Processed 14000/21641 files...
Processed 15000/21641 files...
Processed 16000/21641 files...
Processed 17000/21641 files...
Processed 18000/21641 files...
Processed 19000/21641 files...
Processed 20000/21641 files...
Processed 21000/21641 files...
Finished processing 21641 files.
Saving updated data with reference codes to: swiss_law_footers_with_codes.json
Successfully saved updated JSON file.


In [ ]:
import json
import re
from pathlib import Path
import sys
import logging

# --- Configuration ---
# Input JSON file from the previous step (contains initial reference_codes)
INPUT_JSON_PATH = Path("swiss_law_footers_with_codes.json")
# Output JSON file with the added list-based reference codes
OUTPUT_JSON_PATH = Path("swiss_law_footers_with_codes_final.json")
# Log file for RU list pattern matches
LOG_FILE_PATH = Path("ru_list_pattern_matches.log")

# --- Regex Definitions ---
# Compile the main pattern for finding the lists
# Group 1: FF or RU
# Group 2: First YYYY NNNN
# Group 3: The rest of the list starting with ;
LIST_PATTERN = re.compile(r"\b(FF|RU)\s+(\d{4}\s+\d+)\b((?:;\s*\d{4}\s+\d+\b)*)")
# Simple pattern to extract YYYY NNNN from list fragments
YEAR_NUM_PATTERN = re.compile(r"(\d{4}\s+\d+)")

# --- Setup Logging for RU matches ---
# Configure a specific logger for RU matches to keep the main log clean
ru_logger = logging.getLogger('RU_List_Matcher')
ru_logger.setLevel(logging.INFO)
# Clear previous log file if it exists
if LOG_FILE_PATH.exists():
    LOG_FILE_PATH.unlink()
ru_handler = logging.FileHandler(LOG_FILE_PATH, encoding='utf-8')
ru_formatter = logging.Formatter('%(asctime)s - %(message)s')
ru_handler.setFormatter(ru_formatter)
# Avoid adding handler if it already exists (e.g., during testing/re-running)
if not ru_logger.hasHandlers():
    ru_logger.addHandler(ru_handler)

# --- Main Processing Function ---
def add_list_reference_codes(input_path, output_path):
    """
    Reads the input JSON (which already has 'reference_codes'),
    finds additional codes from semicolon-separated lists (FF/RU YYYY NNNN; YYYY NNNN...),
    merges them into the 'reference_codes' list, and saves to a new JSON file.
    Logs matches starting with RU to a separate file.
    """
    if not input_path.is_file():
        print(f"Error: Input JSON file not found at '{input_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_path}")
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            all_files_data = json.load(f)
        print(f"Loaded data for {len(all_files_data)} files.")
    except Exception as e:
        print(f"Error reading or parsing input JSON file '{input_path}': {e}", file=sys.stderr)
        return

    print("Processing files to extract list-based reference codes...")
    processed_count = 0
    # Iterate through each file's data structure
    for file_data in all_files_data:
        # Get filename for logging
        filename = file_data.get("filename", "Unknown Filename")
        # Get the existing codes (initialize as empty set if missing/not list)
        try:
            # Use set for efficient addition and uniqueness handling
            existing_codes = set(file_data.get("reference_codes", []))
            if not isinstance(file_data.get("reference_codes", []), list):
                 print(f"Warning: 'reference_codes' in '{filename}' was not a list. Initializing empty.", file=sys.stderr)
                 existing_codes = set() # Start fresh if format was wrong
        except TypeError: # Handle non-iterable existing data
             print(f"Warning: 'reference_codes' in '{filename}' was not iterable. Initializing empty.", file=sys.stderr)
             existing_codes = set()


        # Set to store NEW codes found from the list pattern in this file
        additional_codes_found = set()

        references_list = file_data.get("references", [])
        if not isinstance(references_list, list):
            print(f"Warning: 'references' field in '{filename}' is not a list. Skipping list code extraction.", file=sys.stderr)
            continue # Skip processing this file if references aren't a list

        # Iterate through each reference dictionary
        for ref_dict in references_list:
             if not isinstance(ref_dict, dict): continue # Skip non-dict items
             try:
                 footnote_text = list(ref_dict.values())[0]
                 if not isinstance(footnote_text, str): continue # Skip non-string text
             except IndexError: continue # Skip empty dict
             except Exception as e:
                 print(f"Warning: Error extracting footnote text from {ref_dict} in '{filename}' for list check: {e}. Skipping.", file=sys.stderr)
                 continue

             # --- Find List Patterns ---
             # Use finditer to handle multiple list patterns in the same text
             for match in LIST_PATTERN.finditer(footnote_text):
                 try:
                     prefix = match.group(1)          # FF or RU
                     first_year_num = match.group(2)  # First YYYY NNNN
                     rest_of_list = match.group(3)    # String with "; YYYY NNNN..."

                     # Construct and add the first code
                     first_code = f"{prefix} {first_year_num}"
                     additional_codes_found.add(first_code)

                     # Log if it's an RU match
                     if prefix == "RU":
                         ru_logger.info(f"File: {filename} - Found RU List Match: '{match.group(0)}'")
                         ru_logger.info(f"  -> Extracted initial: {first_code}")

                     # Process the rest of the list if it exists
                     if rest_of_list:
                         # Split the rest of the string by semicolon
                         fragments = rest_of_list.split(';')
                         for fragment in fragments:
                             fragment = fragment.strip() # Remove surrounding whitespace
                             if not fragment: continue # Skip empty fragments

                             # Extract the YYYY NNNN from the fragment
                             year_num_match = YEAR_NUM_PATTERN.search(fragment)
                             if year_num_match:
                                 year_num_part = year_num_match.group(1)
                                 # Construct the full code using the original prefix
                                 implied_code = f"{prefix} {year_num_part}"
                                 additional_codes_found.add(implied_code)
                                 # Log the implied RU code
                                 if prefix == "RU":
                                     ru_logger.info(f"  -> Implied from fragment '{fragment}': {implied_code}")
                             else:
                                 # Log if a fragment couldn't be parsed (optional)
                                 # print(f"Debug: Could not parse YYYY NNNN from fragment: '{fragment}' in {filename}")
                                 pass

                 except Exception as e:
                     # Log error processing a specific match
                      print(f"Warning: Error processing list pattern match '{match.group(0)}' in '{filename}': {e}", file=sys.stderr)


        # --- Update File Data ---
        # Combine existing codes with newly found codes
        # The union operation automatically handles duplicates
        combined_codes = existing_codes.union(additional_codes_found)
        # Convert back to a sorted list for the final JSON
        file_data["reference_codes"] = sorted(list(combined_codes))

        processed_count += 1
        if processed_count % 1000 == 0:
             print(f"Processed {processed_count}/{len(all_files_data)} files...")

    print(f"Finished processing {processed_count} files.")

    # --- Save Updated Data ---
    print(f"Saving updated data with combined reference codes to: {output_path}")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(all_files_data, f, ensure_ascii=False, indent=4)
        print("Successfully saved updated JSON file.")
        print(f"Check '{LOG_FILE_PATH}' for details on detected RU list patterns.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_path}': {e}", file=sys.stderr)

# --- Script Execution ---
if __name__ == "__main__":
    add_list_reference_codes(INPUT_JSON_PATH, OUTPUT_JSON_PATH)

### Clean RS to RU


In [6]:
import csv

csv_file = "./raw_data/RS to RU.csv"
cleaned_rows = []

with open(csv_file, newline='', encoding='utf-8') as f:
    # Read raw lines
    raw_lines = f.readlines()

    # Clean and parse each line
    for line in raw_lines:
        # Split by comma, strip spaces and quotes
        cleaned_line = [field.strip().strip('"') for field in line.strip().split(',') if field.strip()]
        if cleaned_line:
            cleaned_rows.append(cleaned_line)

# Now convert to list of dictionaries using the header
header = cleaned_rows[0]
data_rows = cleaned_rows[1:]

structured_data = [dict(zip(header, row)) for row in data_rows]

# save to json
import json
output_json_file = "./raw_data/RS_to_RU.json"
with open(output_json_file, 'w', encoding='utf-8') as f:
    json.dump(structured_data, f, ensure_ascii=False, indent=4)



In [21]:
result_dict = {}

with open(csv_file, newline='', encoding='utf-8') as f:
    raw_lines = f.readlines()

    cleaned_rows = []
    for line in raw_lines:
        cleaned_line = [field.strip().strip('"') for field in line.strip().split(',') if field.strip()]
        if cleaned_line:
            cleaned_rows.append(cleaned_line)

    header = cleaned_rows[0]
    data_rows = cleaned_rows[1:]

    for row in data_rows:
        row_dict = dict(zip(header, row))
        key = f"RS {row_dict['rsNotationStr']}"
        value = row_dict['ruIdentifier']
        result_dict[key] = value

# Print result
import json
output_json_file = "./raw_data/RS_to_RU.json"
with open(output_json_file, 'w', encoding='utf-8') as f:
    json.dump(result_dict, f, ensure_ascii=False, indent=4)


### Using two json, translate RS to RU and gather them all under the filename

In [ ]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
# Input JSON from the previous step (with initial reference_codes)
INPUT_JSON_PATH = Path("swiss_law_footers_with_codes_final.json")
# Input JSON with the RS to RU mapping
RS_TO_RU_MAP_PATH = Path("./raw_data/RS_to_RU.json")
# Final output JSON with cleaned filenames and translated codes
OUTPUT_JSON_PATH = Path("swiss_law_references_translated.json")

# Categories that should use the "FF" prefix (lowercase for case-insensitive matching)
# Note: Add ALL required categories here in lowercase
FF_CATEGORIES = {
    "strategia e obiettivi",
    "rapporto del consiglio federale",
    "istruzioni del consiglio federale",
    "garanzia alle costituzioni cantonali",
    "decreto del consiglio federale",
    "decisioni delle autorità",
    "correzione ff",
    "convenzioni delle aziende e degli stabilimenti autonomi",
    "convezioni del consiglio federale", # Typo in request? Assuming "convenzioni"
    "convenzioni del consiglio federale",
    "concessioni"
}

# Regex to parse the original PDF filename format (YYYY-NNN.pdf)
FILENAME_REGEX = re.compile(r"(\d{4})-(\d+)\.pdf$")

# --- Main Processing Function ---
def clean_and_translate(input_json_path, map_json_path, output_json_path):
    """
    Cleans filenames based on category and translates RS codes using a mapping.
    """
    # --- Load Input Data ---
    if not input_json_path.is_file():
        print(f"Error: Input JSON file not found at '{input_json_path}'", file=sys.stderr)
        return
    if not map_json_path.is_file():
        print(f"Error: RS to RU map file not found at '{map_json_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_json_path}")
    try:
        with open(input_json_path, 'r', encoding='utf-8') as f:
            all_files_data = json.load(f)
        print(f"Loaded data for {len(all_files_data)} files.")
    except Exception as e:
        print(f"Error reading or parsing input JSON file '{input_json_path}': {e}", file=sys.stderr)
        return

    print(f"Loading RS to RU map from: {map_json_path}")
    try:
        with open(map_json_path, 'r', encoding='utf-8') as f:
            rs_to_ru_map = json.load(f)
        print(f"Loaded {len(rs_to_ru_map)} RS code mappings.")
    except Exception as e:
        print(f"Error reading or parsing RS to RU map file '{map_json_path}': {e}", file=sys.stderr)
        return

    # --- Process Data ---
    print("Processing files: cleaning filenames and translating codes...")
    final_results = []
    processed_count = 0

    for file_data in all_files_data:
        original_filename = file_data.get("filename", "")
        category = file_data.get("category", "unknown").lower() # Use lowercase category
        reference_codes = file_data.get("reference_codes", [])

        # Initialize fields for the output object
        cleaned_filename = None
        translated_reference_codes = []
        missing_rs_codes = []

        # --- 1. Clean Filename ---
        match = FILENAME_REGEX.search(original_filename)
        if match:
            year = match.group(1)
            number = match.group(2)
            # Determine prefix based on category
            prefix = "FF" if category in FF_CATEGORIES else "RU"
            cleaned_filename = f"{prefix} {year} {number}"
        else:
            print(f"Warning: Could not parse original filename '{original_filename}'. Skipping cleaning.", file=sys.stderr)
            cleaned_filename = original_filename # Keep original if parsing failed

        # --- 2. Translate Reference Codes ---
        if isinstance(reference_codes, list):
            for code in reference_codes:
                if not isinstance(code, str):
                     print(f"Warning: Non-string code found in '{original_filename}': {code}. Skipping.", file=sys.stderr)
                     continue # Skip non-string codes

                if code.startswith("RS "):
                    # Check if the RS code exists in the mapping
                    if code in rs_to_ru_map:
                        # Add the translated RU code
                        translated_reference_codes.append(rs_to_ru_map[code])
                    else:
                        # RS code not found in map, add to missing list
                        missing_rs_codes.append(code)
                        # Also add the original RS code to the translated list?
                        # Decision: Let's add it to translated list as well to not lose it.
                        translated_reference_codes.append(code)
                else:
                    # Keep FF, RU, or other codes as they are
                    translated_reference_codes.append(code)
        else:
             print(f"Warning: 'reference_codes' in '{original_filename}' is not a list. Translation skipped.", file=sys.stderr)


        # --- 3. Build Output Object ---
        output_data = {
            "original_filename": original_filename,
            "category": file_data.get("category", "unknown"), # Keep original case category here
            "cleaned_filename": cleaned_filename,
            "translated_reference_codes": sorted(translated_reference_codes), # Sort final list
            "missing_rs_codes": sorted(missing_rs_codes) # Sort missing list
            # Optionally include other original fields if needed
            # "number_of_references": file_data.get("number_of_references"),
            # "warnings": file_data.get("warnings"),
        }
        final_results.append(output_data)

        processed_count += 1
        if processed_count % 1000 == 0:
             print(f"Processed {processed_count}/{len(all_files_data)} files...")

    print(f"Finished processing {processed_count} files.")

    # --- Save Final Results ---
    print(f"Saving final translated results to: {output_json_path}")
    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=4)
        print("Successfully saved final JSON file.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_json_path}': {e}", file=sys.stderr)

# --- Script Execution ---
if __name__ == "__main__":
    clean_and_translate(INPUT_JSON_PATH, RS_TO_RU_MAP_PATH, OUTPUT_JSON_PATH)

Loading data from: swiss_law_footers_with_codes_final.json
Loaded data for 21641 files.
Loading RS to RU map from: ./raw_data/RS_to_RU.json
Loaded 8857 RS code mappings.
Processing files: cleaning filenames and translating codes...
Processed 1000/21641 files...
Processed 2000/21641 files...
Processed 3000/21641 files...
Processed 4000/21641 files...
Processed 5000/21641 files...
Processed 6000/21641 files...
Processed 7000/21641 files...
Processed 8000/21641 files...
Processed 9000/21641 files...
Processed 10000/21641 files...
Processed 11000/21641 files...
Processed 12000/21641 files...
Processed 13000/21641 files...
Processed 14000/21641 files...
Processed 15000/21641 files...
Processed 16000/21641 files...
Processed 17000/21641 files...
Processed 18000/21641 files...
Processed 19000/21641 files...
Processed 20000/21641 files...
Processed 21000/21641 files...
Finished processing 21641 files.
Saving final translated results to: swiss_law_references_translated.json


Successfully saved final JSON file.


In [42]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
INPUT_JSON_PATH = Path("./raw_data/RS_to_RU_full.json") # Replace with your input file name
OUTPUT_JSON_MAP_PATH = Path("./raw_data/RS_to_RU_codes.json")
DUPLICATE_LOG_PATH = Path("RS_to_RU_duplicates.log")

# --- Regex Definitions (Copied from previous script) ---
# Since 1999: /oc/YYYY/NNNN
RE_SINCE_1999 = re.compile(r"/oc/(\d{4})/(\d+)$")
# 1948-1998: /oc/YYYY/page1_page2_page3 (pages optional)
RE_1948_1998 = re.compile(r"/oc/(\d{4})/((\d*)_(\d*)_(\d*))$")
# Pre-1948: /oc/Volume/page1_page2_page3 (pages optional)
RE_PRE_1948 = re.compile(r"/oc/([IVXLCDM]+|\d+)/((\d*)_(\d*)_(\d*))$")

# --- Helper Function: URI Parser ---
def parse_uri_to_ru_id(uri_string):
    """
    Parses a potential RU Act URI and returns the formatted RU ID string
    (e.g., "RU YYYY NNNN" or "RU Vol Page") or None if parsing fails.
    """
    if not uri_string or not isinstance(uri_string, str):
        return None # Invalid input

    extracted_id_part = None # Will hold "YYYY NNNN" or "Vol Page"
    prefix = "RU" # Assuming RU prefix based on requirements

    # Rule 1: Since 1999
    match1 = RE_SINCE_1999.search(uri_string)
    if match1:
        year, identifier = match1.groups()
        try:
             if int(year) >= 1999:
                 extracted_id_part = f"{year} {identifier}"
        except ValueError: pass # Ignore if year is not an int

    # Rule 2: 1948-1998
    if not extracted_id_part:
        match2 = RE_1948_1998.search(uri_string)
        if match2:
            year, _, page1, page2, page3 = match2.groups()
            try:
                if 1948 <= int(year) <= 1998:
                    last_page = page3 if page3 else (page2 if page2 else page1)
                    if last_page:
                        extracted_id_part = f"{year} {last_page}"
                    # else: implicit None if no page found
            except ValueError: pass # Ignore if year is not int

    # Rule 3: Pre-1948
    if not extracted_id_part:
        match3 = RE_PRE_1948.search(uri_string)
        if match3:
            volume, _, page1, page2, page3 = match3.groups()
            last_page = page3 if page3 else (page2 if page2 else page1)
            if last_page:
                extracted_id_part = f"{volume} {last_page}"
            # else: implicit None if no page found

    # Format the final string or return None
    if extracted_id_part:
        return f"{prefix} {extracted_id_part}"
    else:
        return None # Return None if no pattern matched or parsing failed

# --- Main Transformation Function ---
def create_rs_to_ru_map(input_path, output_path, log_path):
    """
    Reads input JSON, creates an RS -> RU ID map, logs duplicates,
    and saves the final map.
    """
    if not input_path.is_file():
        print(f"Error: Input JSON file not found at '{input_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_path}")
    try:
        with open(input_path, 'r', encoding='utf-8') as f: input_data = json.load(f)
        if not isinstance(input_data, list):
             print(f"Error: Input JSON does not contain a list.", file=sys.stderr); return
        print(f"Loaded {len(input_data)} records.")
    except Exception as e: print(f"Error reading or parsing input JSON file '{input_path}': {e}", file=sys.stderr); return

    # Dictionary for the final RS -> RU mapping
    final_rs_to_ru_map = {}
    # List to log overwritten duplicate entries
    duplicate_log_entries = []
    print("Creating RS -> RU map...")
    record_count = 0

    for record in input_data:
        record_count += 1
        if not isinstance(record, dict):
            print(f"Warning [Record {record_count}]: Skipping non-dictionary item: {record}", file=sys.stderr); continue

        rs_notation_raw = record.get("rsNotationStr")
        potential_ru_uri = record.get("potentialRuActURI")

        # --- Validate and Prepare RS Key ---
        if not isinstance(rs_notation_raw, str):
            print(f"Warning [Record {record_count}]: Skipping record due to invalid 'rsNotationStr' type ({type(rs_notation_raw)}): {record}", file=sys.stderr); continue
        rs_notation_trimmed = rs_notation_raw.strip()
        # Basic check if it looks like a valid number sequence after trimming
        if not rs_notation_trimmed or not re.match(r'^[\d.]+$', rs_notation_trimmed):
            print(f"Warning [Record {record_count}]: Skipping record due to invalid 'rsNotationStr' value: '{rs_notation_raw}'", file=sys.stderr); continue
        # Construct the final key
        rs_key = f"RS {rs_notation_trimmed}"

        # --- Parse URI to get RU ID (Value) ---
        if not potential_ru_uri or not isinstance(potential_ru_uri, str):
             print(f"Warning [Record {record_count}, Key '{rs_key}']: Skipping due to missing or invalid 'potentialRuActURI': {record}", file=sys.stderr); continue
        ru_id = parse_uri_to_ru_id(potential_ru_uri)
        if ru_id is None:
            print(f"Warning [Record {record_count}, Key '{rs_key}']: Could not parse RU ID from URI '{potential_ru_uri}'. Skipping entry.", file=sys.stderr); continue

        # --- Handle Duplicates and Update Map ---
        if rs_key in final_rs_to_ru_map:
            existing_ru_id = final_rs_to_ru_map[rs_key]
            # Only log if the new RU ID is actually different
            if ru_id != existing_ru_id:
                duplicate_log_entries.append({
                    "rs_key": rs_key,
                    "overwritten_ru_id": existing_ru_id,
                    "new_ru_id": ru_id,
                    "record_number": record_count # Track where the overwrite happened
                })
                print(f"Info [Record {record_count}]: Overwriting key '{rs_key}'. Old: '{existing_ru_id}', New: '{ru_id}'")
            # Always update with the latest value found in the input file
            final_rs_to_ru_map[rs_key] = ru_id
        else:
            # Key doesn't exist, add it
            final_rs_to_ru_map[rs_key] = ru_id

    print(f"Finished processing {record_count} records.")
    print(f"Final map contains {len(final_rs_to_ru_map)} unique RS keys.")

    # --- Save Output Map ---
    print(f"Saving final RS -> RU map to: {output_path}")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_rs_to_ru_map, f, ensure_ascii=False, indent=4, sort_keys=True) # Sort keys for consistency
        print("Successfully saved final map JSON file.")
    except Exception as e:
        print(f"Error writing output JSON map file '{output_path}': {e}", file=sys.stderr)

    # --- Save Duplicate Log ---
    if duplicate_log_entries:
        print(f"Saving duplicate log ({len(duplicate_log_entries)} entries) to: {log_path}")
        try:
            with open(log_path, 'w', encoding='utf-8') as f:
                for entry in duplicate_log_entries:
                    # Write as JSON lines for easy parsing later if needed
                    json.dump(entry, f, ensure_ascii=False)
                    f.write('\n')
            print("Successfully saved duplicate log file.")
        except Exception as e:
            print(f"Error writing duplicate log file '{log_path}': {e}", file=sys.stderr)
    else:
        print("No duplicate RS keys with different RU IDs were overwritten.")


# --- Script Execution ---
if __name__ == "__main__":
    # Create dummy input file if it doesn't exist
    if not INPUT_JSON_PATH.exists():
        print(f"Creating dummy input file: {INPUT_JSON_PATH}")
        dummy_data = [
            {"rsNotationStr": "0.101", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/1974/2151_2151_2151", "ruIdentifier": "..."},
            {"rsNotationStr": "0.101.02", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/1974/2175_2175_2175", "ruIdentifier": "..."},
            {"rsNotationStr": "0.101.2", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/2006/596", "ruIdentifier": "..."},
            {"rsNotationStr": "0.101.09", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/1998/2993_2993_2993", "ruIdentifier": "..."}, # Duplicate 1
            {"rsNotationStr": "0.101.09", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/1995/3950_3950_3950", "ruIdentifier": "..."}, # Duplicate 2 (kept)
            {"rsNotationStr": " ", "rsAbstractURI": "...", "potentialRuActURI": "https://fedlex.data.admin.ch/eli/oc/2000/10", "ruIdentifier": "..."}, # Invalid rsNotationStr
            {"rsNotationStr": "0.103.1", "rsAbstractURI": "...", "potentialRuActURI": "https://invalid_uri", "ruIdentifier": "..."}, # Invalid URI
        ]
        with open(INPUT_JSON_PATH, 'w', encoding='utf-8') as f_dummy:
             json.dump(dummy_data, f_dummy, indent=4)

    create_rs_to_ru_map(INPUT_JSON_PATH, OUTPUT_JSON_MAP_PATH, DUPLICATE_LOG_PATH)

Loading data from: ./raw_data/RS_to_RU_full.json
Loaded 13006 records.
Creating RS -> RU map...
Info [Record 7]: Overwriting key 'RS 0.101.09'. Old: 'RU 1998 2993', New: 'RU 1995 3950'
Info [Record 13]: Overwriting key 'RS 0.101.2'. Old: 'RU 2006 596', New: 'RU 2000 10'
Info [Record 104]: Overwriting key 'RS 0.141.134.92'. Old: 'RU 1998 89', New: 'RU 1959 223'
Info [Record 106]: Overwriting key 'RS 0.141.134.921'. Old: 'RU 1959 227', New: 'RU 2002 654'
Info [Record 148]: Overwriting key 'RS 0.142.111.639'. Old: 'RU 2003 399', New: 'RU 1955 61'
Info [Record 163]: Overwriting key 'RS 0.142.111.919'. Old: 'RU 2009 452', New: 'RU 2005 355'
Info [Record 172]: Overwriting key 'RS 0.142.112.149'. Old: 'RU 2009 306', New: 'RU 1994 1852'
Info [Record 188]: Overwriting key 'RS 0.142.112.739'. Old: 'RU 2011 631', New: 'RU 2008 485'
Info [Record 209]: Overwriting key 'RS 0.142.113.328.1'. Old: 'RU 1989 2551', New: 'RU 1989 2038'
Info [Record 226]: Overwriting key 'RS 0.142.113.499'. Old: 'RU 2003 

In [41]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
INPUT_JSON_PATH = Path("swiss_law_footers_with_codes_final.json") # Input from previous list extraction
RS_TO_RU_MAP_PATH = Path("./raw_data/RS_to_RU_codes.json")
OUTPUT_JSON_PATH = Path("swiss_law_references_translated_v3.json") # New output filename

# Categories that should use the "FF" prefix (lowercase)
FF_CATEGORIES = {
    "strategia e obiettivi", "rapporto del consiglio federale",
    "istruzioni del consiglio federale", "garanzia alle costituzioni cantonali",
    "decreto del consiglio federale", "decisioni delle autorità",
    "correzione ff", "convenzioni delle aziende e degli stabilimenti autonomi",
    "convenzioni del consiglio federale", # Corrected potential typo
    "concessioni"
}

# --- Regex Definitions ---
# Regex for YYYY-NNN_..._NNN.pdf format
# Captures Year (group 1) and the *last* Number (group 2)
FILENAME_REGEX_MULTI_NUM = re.compile(r"(\d{4})-\d+(?:_\d+)*_(\d+)\.pdf$")
# Regex for the simpler YYYY-NNN.pdf format
# Captures Year (group 1) and Number (group 2)
FILENAME_REGEX_SINGLE_NUM = re.compile(r"(\d{4})-(\d+)\.pdf$")

# --- Main Processing Function ---
def clean_and_translate(input_json_path, map_json_path, output_json_path):
    """
    Cleans filenames (handling single and multi-number formats) based on category
    and translates RS codes using a mapping.
    """
    # --- Load Input Data ---
    if not input_json_path.is_file(): print(f"Error: Input JSON file not found at '{input_json_path}'", file=sys.stderr); return
    if not map_json_path.is_file(): print(f"Error: RS to RU map file not found at '{map_json_path}'", file=sys.stderr); return

    print(f"Loading data from: {input_json_path}")
    try:
        with open(input_json_path, 'r', encoding='utf-8') as f: all_files_data = json.load(f)
        print(f"Loaded data for {len(all_files_data)} files.")
    except Exception as e: print(f"Error reading or parsing input JSON file '{input_json_path}': {e}", file=sys.stderr); return

    print(f"Loading RS to RU map from: {map_json_path}")
    try:
        with open(map_json_path, 'r', encoding='utf-8') as f: rs_to_ru_map = json.load(f)
        print(f"Loaded {len(rs_to_ru_map)} RS code mappings.")
    except Exception as e: print(f"Error reading or parsing RS to RU map file '{map_json_path}': {e}", file=sys.stderr); return

    # --- Process Data ---
    print("Processing files: cleaning filenames and translating codes...")
    final_results = []
    processed_count = 0

    for file_data in all_files_data:
        original_filename = file_data.get("filename", "")
        # Get category and convert to lowercase for matching FF_CATEGORIES
        category = file_data.get("category", "unknown").lower()
        reference_codes = file_data.get("reference_codes", [])

        cleaned_filename = None
        year = None
        number = None

        # --- 1. Clean Filename (Revised Logic) ---
        # Try matching the multi-number format first
        match_multi = FILENAME_REGEX_MULTI_NUM.search(original_filename)
        if match_multi:
            year = match_multi.group(1)
            number = match_multi.group(2) # Use the last number captured
            # print(f"DEBUG: Matched MULTI: {original_filename} -> Year={year}, Number={number}") # Optional debug
        else:
            # If multi-number didn't match, try the single-number format
            match_single = FILENAME_REGEX_SINGLE_NUM.search(original_filename)
            if match_single:
                year = match_single.group(1)
                number = match_single.group(2)
                # print(f"DEBUG: Matched SINGLE: {original_filename} -> Year={year}, Number={number}") # Optional debug

        # If year and number were successfully extracted by either regex
        if year and number:
            prefix = "FF" if category in FF_CATEGORIES else "RU"
            cleaned_filename = f"{prefix} {year} {number}"
        else:
            # If neither regex matched, log warning and use original filename
            print(f"Warning: Could not parse year/number from original filename '{original_filename}'. Using original.", file=sys.stderr)
            cleaned_filename = original_filename

        # --- 2. Translate Reference Codes (Same as before) ---
        translated_reference_codes = []
        missing_rs_codes = []
        if isinstance(reference_codes, list):
            for code in reference_codes:
                if not isinstance(code, str):
                     print(f"Warning: Non-string code found in '{original_filename}': {code}. Skipping.", file=sys.stderr)
                     continue
                if code.startswith("RS "):
                    if code in rs_to_ru_map:
                        translated_reference_codes.append(rs_to_ru_map[code])
                    else:
                        missing_rs_codes.append(code)
                        translated_reference_codes.append(code) # Keep original RS if not found
                else:
                    translated_reference_codes.append(code) # Keep FF/RU etc.
        else:
             print(f"Warning: 'reference_codes' in '{original_filename}' is not a list. Translation skipped.", file=sys.stderr)

        # --- 3. Build Output Object ---
        output_data = {
            "original_filename": original_filename,
            "category": file_data.get("category", "unknown"), # Keep original case category
            "cleaned_filename": cleaned_filename,
            "translated_reference_codes": sorted(translated_reference_codes),
            "missing_rs_codes": sorted(missing_rs_codes)
        }
        final_results.append(output_data)

        processed_count += 1
        if processed_count % 1000 == 0:
             print(f"Processed {processed_count}/{len(all_files_data)} files...")

    print(f"Finished processing {processed_count} files.")

    # --- Save Final Results ---
    print(f"Saving final translated results to: {output_json_path}")
    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=4)
        print("Successfully saved final JSON file.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_json_path}': {e}", file=sys.stderr)

# --- Script Execution ---
if __name__ == "__main__":
    clean_and_translate(INPUT_JSON_PATH, RS_TO_RU_MAP_PATH, OUTPUT_JSON_PATH)

Loading data from: swiss_law_footers_with_codes_final.json
Loaded data for 21641 files.
Loading RS to RU map from: ./raw_data/RS_to_RU_codes.json
Loaded 8857 RS code mappings.
Processing files: cleaning filenames and translating codes...
Processed 1000/21641 files...
Processed 2000/21641 files...
Processed 3000/21641 files...
Processed 4000/21641 files...
Processed 5000/21641 files...
Processed 6000/21641 files...
Processed 7000/21641 files...
Processed 8000/21641 files...
Processed 9000/21641 files...
Processed 10000/21641 files...
Processed 11000/21641 files...
Processed 12000/21641 files...
Processed 13000/21641 files...
Processed 14000/21641 files...
Processed 15000/21641 files...
Processed 16000/21641 files...
Processed 17000/21641 files...
Processed 18000/21641 files...
Processed 19000/21641 files...
Processed 20000/21641 files...
Processed 21000/21641 files...
Finished processing 21641 files.
Saving final translated results to: swiss_law_references_translated_v3.json
Successfull

In [27]:
import json
from pathlib import Path
import sys

# --- Configuration ---
# Path to the final JSON file generated by the previous script
FINAL_JSON_PATH = Path("swiss_law_references_translated_v2.json")

# --- Main Checking Function ---
def check_for_missing_rs(json_path):
    """
    Reads the final JSON file and checks for entries with missing RS codes.
    Prints a summary report.
    """
    # Check if the input file exists
    if not json_path.is_file():
        print(f"Error: Final JSON file not found at '{json_path}'", file=sys.stderr)
        return

    print(f"Checking for missing RS codes in: {json_path}")

    # Initialize counters and storage for details
    files_with_missing_count = 0
    total_files_checked = 0
    missing_details = [] # List to store info about files with missing codes
    unique_missing_codes = set() # To store all unique missing RS codes found

    # Read and parse the JSON file
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            all_files_data = json.load(f)
        total_files_checked = len(all_files_data)
        print(f"Successfully loaded data for {total_files_checked} files.")
    except json.JSONDecodeError as e:
        print(f"Error: Failed to decode JSON from '{json_path}'. Invalid JSON format. Details: {e}", file=sys.stderr)
        return
    except Exception as e:
        print(f"Error reading file '{json_path}': {e}", file=sys.stderr)
        return

    # Iterate through each processed file record
    for file_data in all_files_data:
        # Safely get the list of missing codes, default to empty list
        missing_codes = file_data.get("missing_rs_codes", [])

        # Check if it's a non-empty list
        if isinstance(missing_codes, list) and missing_codes:
            files_with_missing_count += 1
            cleaned_filename = file_data.get("cleaned_filename", "N/A")
            original_filename = file_data.get("original_filename", "N/A")

            # Store details for reporting
            missing_details.append({
                "cleaned_filename": cleaned_filename,
                "original_filename": original_filename,
                "missing": missing_codes
            })
            # Add the missing codes to the set of unique missing codes
            unique_missing_codes.update(missing_codes)

    # --- Print Summary Report ---
    print("\n--- Missing RS Code Analysis Report ---")
    print(f"Total files checked: {total_files_checked}")

    if files_with_missing_count == 0:
        print("No files found with missing RS codes.")
    else:
        print(f"Files with missing RS codes: {files_with_missing_count}")
        print("-" * 40)
        print("List of Unique Missing RS Codes Found:")
        if unique_missing_codes:
            for code in sorted(list(unique_missing_codes)):
                print(f"  - {code}")
        else:
             print("  (No unique codes recorded - check logic)") # Should not happen if count > 0

        print("-" * 40)
        # Optionally print details for each file (can be long)
        # Set PRINT_DETAILS to True if you want the full list
        PRINT_DETAILS = False # Change to True to see details for every file
        if PRINT_DETAILS:
            print("Details of Files with Missing RS Codes:")
            for detail in missing_details:
                print(f"  File (Cleaned): {detail['cleaned_filename']}")
                print(f"  File (Original): {detail['original_filename']}")
                print(f"  Missing Codes: {', '.join(detail['missing'])}")
                print("  ---")
        else:
             print("Set PRINT_DETAILS = True in the script to list details for each file.")


    print("-" * 40)

# --- Script Execution ---
if __name__ == "__main__":
    check_for_missing_rs(FINAL_JSON_PATH)

Checking for missing RS codes in: swiss_law_references_translated_v2.json
Successfully loaded data for 21641 files.

--- Missing RS Code Analysis Report ---
Total files checked: 21641
Files with missing RS codes: 166
----------------------------------------
List of Unique Missing RS Codes Found:
  - RS 0
  - RS 0.142
  - RS 0.142.395.41
  - RS 0.232.151
  - RS 0.232.171.1
  - RS 0.311
  - RS 0.360.268.1
  - RS 0.360.268.10
  - RS 0.360.268.11
  - RS 0.360.268.121.0
  - RS 0.360.268.2
  - RS 0.360.314.1
  - RS 0.360.598.1
  - RS 0.361.191.1
  - RS 0.362.380.045
  - RS 0.420.514.291
  - RS 0.632
  - RS 0.632.312.451.1
  - RS 0.632.312.491
  - RS 0.632.312.741
  - RS 0.632.312.741.1
  - RS 0.632.312.811.1
  - RS 0.632.312.911.1
  - RS 0.632.313.141
  - RS 0.632.313.211.1
  - RS 0.632.314.671.1
  - RS 0.632.314.891.1
  - RS 0.632.315.201.11
  - RS 0.632.315.491.1
  - RS 0.632.316.251.1
  - RS 0.632.316.421
  - RS 0.632.316.421.1
  - RS 0.632.317.581.1
  - RS 0.632.402.81
  - RS 0.632.402.8

In [35]:
csv_file = "./raw_data/RS to RU.csv"
cleaned_rows = []

with open(csv_file, newline='', encoding='utf-8') as f:
    # Read raw lines
    raw_lines = f.readlines()

    # Clean and parse each line
    for line in raw_lines:
        # Split by comma, strip spaces and quotes
        cleaned_line = [field.strip().strip('"') for field in line.strip().split(',') if field.strip()]
        if cleaned_line:
            cleaned_rows.append(cleaned_line)

# Now convert to list of dictionaries using the header
header = cleaned_rows[0]
data_rows = cleaned_rows[1:]

structured_data = [dict(zip(header, row)) for row in data_rows]

# save to json
import json
output_json_file = "./raw_data/RS_to_RU_full.json"
with open(output_json_file, 'w', encoding='utf-8') as f:
    json.dump(structured_data, f, ensure_ascii=False, indent=4)



### Clean RU URIs and Types csv to json

In [ ]:
csv_file = "./raw_data/RU_URI and Type.csv"
cleaned_rows = []

with open(csv_file, newline='', encoding='utf-8') as f:
    # Read raw lines
    raw_lines = f.readlines()

    # Clean and parse each line
    for line in raw_lines:
        # Split by comma, strip spaces and quotes
        cleaned_line = [field.strip().strip('"') for field in line.strip().split(',') if field.strip()]
        if cleaned_line:
            cleaned_rows.append(cleaned_line)

# Now convert to list of dictionaries using the header
header = cleaned_rows[0]
data_rows = cleaned_rows[1:]

structured_data = [dict(zip(header, row)) for row in data_rows]

# save to json
import json
output_json_file = "./raw_data/RU_URI and Types.json"
with open(output_json_file, 'w', encoding='utf-8') as f:
    json.dump(structured_data, f, ensure_ascii=False, indent=4)



### Extract codes from the URI

In [30]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
INPUT_JSON_PATH = Path("./raw_data/RU_URI and Types.json")
OUTPUT_JSON_PATH = Path("./raw_data/RU_Id and Types.json")

# Regex Definitions ---
# Since 1999: /oc/YYYY/NNNN
RE_SINCE_1999 = re.compile(r"/oc/(\d{4})/(\d+)$")

# 1948-1998: /oc/YYYY/page1_page2_page3 (pages optional)
# Group 1: Year
# Group 2: Full page string (e.g., "1506_1506_1506", "_1506_", "__")
# Group 3: Page 1 (German, optional digits)
# Group 4: Page 2 (French, optional digits)
# Group 5: Page 3 (Italian, optional digits)
RE_1948_1998 = re.compile(r"/oc/(\d{4})/((\d*)_(\d*)_(\d*))$")

# Pre-1948: /oc/Volume/page1_page2_page3 (pages optional)
# Group 1: Volume (Roman or Arabic)
# Group 2: Full page string
# Group 3: Page 1 (German, optional digits)
# Group 4: Page 2 (French, optional digits)
# Group 5: Page 3 (Italian, optional digits)
RE_PRE_1948 = re.compile(r"/oc/([IVXLCDM]+|\d+)/((\d*)_(\d*)_(\d*))$")

# --- Main Transformation Function ---
def transform_uris_to_ids(input_path, output_path):
    """
    Reads a JSON file with actURIs, extracts identifiers handling missing page numbers,
    and saves a new JSON file with actIds.
    """
    if not input_path.is_file():
        print(f"Error: Input JSON file not found at '{input_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_path}")
    try:
        with open(input_path, 'r', encoding='utf-8') as f: input_data = json.load(f)
        if not isinstance(input_data, list):
             print(f"Error: Input JSON does not contain a list.", file=sys.stderr); return
        print(f"Loaded {len(input_data)} records.")
    except Exception as e: print(f"Error reading or parsing input JSON file '{input_path}': {e}", file=sys.stderr); return

    transformed_results = []
    print("Transforming URIs to IDs...")
    processed_count = 0

    for record in input_data:
        if not isinstance(record, dict):
             print(f"Warning: Skipping non-dictionary item in input: {record}", file=sys.stderr); continue
        act_uri = record.get("actURI")
        category_label = record.get("categoryLabel")
        if not act_uri or not isinstance(act_uri, str):
             print(f"Warning: Skipping record due to missing or invalid 'actURI': {record}", file=sys.stderr); continue
        if category_label is None:
             print(f"Warning: Record missing 'categoryLabel': {record}", file=sys.stderr); category_label = "Unknown"

        extracted_id = None
        found_match = False

        # --- Apply Rules Sequentially ---

        # Rule 1: Since 1999
        match1 = RE_SINCE_1999.search(act_uri)
        if match1:
            year, identifier = match1.groups()
            if int(year) >= 1999:
                extracted_id = f"RU {year} {identifier}"; found_match = True

        # Rule 2: 1948-1998 (Optional pages)
        if not found_match:
            match2 = RE_1948_1998.search(act_uri)
            if match2:
                year = match2.group(1)
                page_string = match2.group(2) # Full page string
                page1 = match2.group(3)       # German page (optional)
                page2 = match2.group(4)       # French page (optional)
                page3 = match2.group(5)       # Italian page (optional)

                if 1948 <= int(year) <= 1998:
                    # Find the last non-empty page number, prioritizing Italian
                    last_page = page3 if page3 else (page2 if page2 else page1)
                    if last_page: # Check if we found at least one page number
                        extracted_id = f"RU {year} {last_page}"; found_match = True
                    else:
                        print(f"Warning: All page numbers missing in URI '{act_uri}'. Skipping.", file=sys.stderr)
                        extracted_id = f"RU {year} UNKNOWN_PAGE" # Assign specific placeholder
                        found_match = True # Mark as handled, even if skipped

        # Rule 3: Pre-1948 (Optional pages)
        if not found_match:
            match3 = RE_PRE_1948.search(act_uri)
            if match3:
                volume = match3.group(1)
                page_string = match3.group(2) # Full page string
                page1 = match3.group(3)       # German page (optional)
                page2 = match3.group(4)       # French page (optional)
                page3 = match3.group(5)       # Italian page (optional)

                # Find the last non-empty page number, prioritizing Italian
                last_page = page3 if page3 else (page2 if page2 else page1)
                if last_page: # Check if we found at least one page number
                    extracted_id = f"RU {volume} {last_page}"; found_match = True
                else:
                    print(f"Warning: All page numbers missing in URI '{act_uri}'. Skipping.", file=sys.stderr)
                    extracted_id = f"RU {volume} UNKNOWN_PAGE" # Assign specific placeholder
                    found_match = True # Mark as handled

        # --- Handle Unmatched URIs ---
        if not found_match:
            extracted_id = "UNKNOWN_URI_FORMAT"
            print(f"Warning: Could not parse URI according to any rule: '{act_uri}'", file=sys.stderr)

        # --- Append Transformed Record ---
        transformed_results.append({"actId": extracted_id, "categoryLabel": category_label})
        processed_count += 1
        if processed_count % 5000 == 0: # Log progress
            print(f"Processed {processed_count}/{len(input_data)} records...")

    print(f"Finished transforming {len(transformed_results)} records.")

    # --- Save Output Data ---
    print(f"Saving transformed data to: {output_path}")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(transformed_results, f, ensure_ascii=False, indent=4)
        print("Successfully saved transformed JSON file.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_path}': {e}", file=sys.stderr)

# --- Script Execution ---
if __name__ == "__main__":
    transform_uris_to_ids(INPUT_JSON_PATH, OUTPUT_JSON_PATH)

Loading data from: ./raw_data/RU_URI and Types.json
Loaded 48347 records.
Transforming URIs to IDs...
Processed 5000/48347 records...
Processed 10000/48347 records...
Processed 15000/48347 records...
Processed 20000/48347 records...
Processed 25000/48347 records...
Processed 30000/48347 records...
Processed 35000/48347 records...
Processed 40000/48347 records...
Processed 45000/48347 records...
Finished transforming 48347 records.
Saving transformed data to: ./raw_data/RU_Id and Types.json
Successfully saved transformed JSON file.


### Clean the RS URI db

In [38]:
import json
import re
from pathlib import Path
import sys

# --- Configuration ---
INPUT_JSON_PATH = Path("./raw_data/RS_to_RU_full.json") # Replace with your input file name (e.g., the sample you provided)
OUTPUT_JSON_PATH = Path("./raw_data/RS_to_RU_codes.json")

# --- Regex Definitions (Reusing from previous script) ---
# Pattern for URIs since 1999: /cc/YYYY/NNNN (Note: Uses /cc/ not /oc/ for RS)
RE_SINCE_1999 = re.compile(r"/cc/(\d{4})/(\d+)$")

# Pattern for URIs 1948-1998: /cc/YYYY/page1_page2_page3 (pages optional)
RE_1948_1998 = re.compile(r"/cc/(\d{4})/((\d*)_(\d*)_(\d*))$")

# Pattern for URIs pre-1948: /cc/Volume/page1_page2_page3 (pages optional)
RE_PRE_1948 = re.compile(r"/cc/([IVXLCDM]+|\d+)/((\d*)_(\d*)_(\d*))$")

# --- Main Transformation Function ---
def create_rs_notation_to_id_map(input_path, output_path):
    """
    Reads JSON with rsNotationStr and rsAbstractURI, extracts RS identifiers
    from the URI, and creates a dictionary mapping Notation -> RS ID.
    Handles duplicate notations by keeping the last encountered value.
    """
    if not input_path.is_file():
        print(f"Error: Input JSON file not found at '{input_path}'", file=sys.stderr)
        return

    print(f"Loading data from: {input_path}")
    try:
        with open(input_path, 'r', encoding='utf-8') as f: input_data = json.load(f)
        if not isinstance(input_data, list):
             print(f"Error: Input JSON does not contain a list.", file=sys.stderr); return
        print(f"Loaded {len(input_data)} records.")
    except Exception as e: print(f"Error reading or parsing input JSON file '{input_path}': {e}", file=sys.stderr); return

    # Dictionary to store the final mapping
    rs_notation_to_id_map = {}
    print("Transforming URIs to create RS Notation -> RS ID map...")
    processed_count = 0

    for record in input_data:
        if not isinstance(record, dict):
             print(f"Warning: Skipping non-dictionary item in input: {record}", file=sys.stderr); continue

        # Extract required fields
        rs_notation = record.get("rsNotationStr")
        uri = record.get("rsAbstractURI") # Use rsAbstractURI

        # Validate fields
        if not rs_notation or not isinstance(rs_notation, str):
             print(f"Warning: Skipping record due to missing or invalid 'rsNotationStr': {record}", file=sys.stderr); continue
        if not uri or not isinstance(uri, str):
             print(f"Warning: Skipping record due to missing or invalid 'rsAbstractURI': {record}", file=sys.stderr); continue

        extracted_id = None
        found_match = False

        # --- Apply URI Parsing Rules ---
        # NOTE: Using "RS" prefix for the output ID now

        # Rule 1: Since 1999 (/cc/YYYY/NNNN)
        match1 = RE_SINCE_1999.search(uri)
        if match1:
            year, identifier = match1.groups()
            if int(year) >= 1999:
                extracted_id = f"RS {year} {identifier}"; found_match = True # Prefix is RS

        # Rule 2: 1948-1998 (/cc/YYYY/page1_page2_page3)
        if not found_match:
            match2 = RE_1948_1998.search(uri)
            if match2:
                year = match2.group(1); page1 = match2.group(3); page2 = match2.group(4); page3 = match2.group(5)
                if 1948 <= int(year) <= 1998:
                    last_page = page3 if page3 else (page2 if page2 else page1)
                    if last_page:
                        extracted_id = f"RS {year} {last_page}"; found_match = True # Prefix is RS
                    else:
                        print(f"Warning: All page numbers missing in URI '{uri}' for RS Notation '{rs_notation}'. Using placeholder.", file=sys.stderr)
                        extracted_id = f"RS {year} UNKNOWN_PAGE"; found_match = True

        # Rule 3: Pre-1948 (/cc/Volume/page1_page2_page3)
        if not found_match:
            match3 = RE_PRE_1948.search(uri)
            if match3:
                volume = match3.group(1); page1 = match3.group(3); page2 = match3.group(4); page3 = match3.group(5)
                last_page = page3 if page3 else (page2 if page2 else page1)
                if last_page:
                    extracted_id = f"RS {volume} {last_page}"; found_match = True # Prefix is RS
                else:
                    print(f"Warning: All page numbers missing in URI '{uri}' for RS Notation '{rs_notation}'. Using placeholder.", file=sys.stderr)
                    extracted_id = f"RS {volume} UNKNOWN_PAGE"; found_match = True

        # --- Handle Unmatched URIs ---
        if not found_match:
            extracted_id = "UNKNOWN_URI_FORMAT"
            print(f"Warning: Could not parse URI '{uri}' for RS Notation '{rs_notation}'. Assigning placeholder.", file=sys.stderr)

        # --- Store in Output Dictionary ---
        # Assign the value. If rs_notation already exists, this overwrites it (keeps last).
        rs_notation_to_id_map[rs_notation] = extracted_id
        processed_count += 1
        if processed_count % 5000 == 0: # Log progress
             print(f"Processed {processed_count}/{len(input_data)} records...")


    print(f"Finished transforming {len(input_data)} records into {len(rs_notation_to_id_map)} map entries.")

    # --- Save Output Dictionary ---
    print(f"Saving RS Notation -> RS ID map to: {output_path}")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            # Dump the dictionary directly
            json.dump(rs_notation_to_id_map, f, ensure_ascii=False, indent=4, sort_keys=True) # Sort keys for consistent output
        print("Successfully saved transformed JSON map file.")
    except Exception as e:
        print(f"Error writing output JSON file '{output_path}': {e}", file=sys.stderr)

# --- Script Execution ---
if __name__ == "__main__":
    # Create a dummy input file for testing if it doesn't exist
    if not INPUT_JSON_PATH.exists():
         print(f"Creating dummy input file: {INPUT_JSON_PATH}")
         dummy_data = [
             {"rsNotationStr": "0.101", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/1974/2151_2151_2151"},
             {"rsNotationStr": "0.101.09", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/1998/2993_2993_2993"},
             {"rsNotationStr": "0.101.09", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/1995/3950_3950_3950"}, # Duplicate Key
             {"rsNotationStr": "0.101.093", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/2003/396"},
             {"rsNotationStr": "101", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/1/116_97_116"}, # Pre-1948 example
             {"rsNotationStr": "10.1", "rsAbstractURI": "https://fedlex.data.admin.ch/eli/cc/1950/__630"}, # Missing page example
             {"rsNotationStr": "10.2", "rsAbstractURI": "https://bad.uri/format"}, # Bad URI example
         ]
         with open(INPUT_JSON_PATH, 'w', encoding='utf-8') as f_dummy:
             json.dump(dummy_data, f_dummy, indent=4)

    create_rs_notation_to_id_map(INPUT_JSON_PATH, OUTPUT_JSON_PATH)

Loading data from: ./raw_data/RS_to_RU_full.json
Loaded 13006 records.
Transforming URIs to create RS Notation -> RS ID map...
Processed 5000/13006 records...
Processed 10000/13006 records...
Finished transforming 13006 records into 8857 map entries.
Saving RS Notation -> RS ID map to: ./raw_data/RS_to_RU_codes.json
Successfully saved transformed JSON map file.
